# Sentiment discrepancy analysis: does the written text support a 4.5-star average?

This notebook is the narrative artifact for the account manager. We compare the
business's quantitative ratings with the qualitative sentiment inferred from each
written review, then investigate the service-language false-negative risk.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from app import MODEL_ID, MODEL_REVISION, build_classifier, enrich_reviews

raw_path = ROOT / "data" / "raw" / "reviews.csv"
reviews = pd.read_csv(raw_path)
reviews.info()
print(f"Rows: {len(reviews):,}; missing values:\n{reviews.isna().sum()}")
print(f"Human rating mean: {reviews['rating'].mean():.2f} / 5")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=reviews, x="rating", ax=axes[0], color="#4c78a8")
axes[0].set_title("Human rating distribution")
reviews["text_length"] = reviews["review_text"].fillna("").astype(str).str.len()
sns.histplot(data=reviews, x="text_length", bins=20, ax=axes[1], color="#f58518")
axes[1].set_title("Review text length distribution")
axes[1].set_xlabel("Characters")
plt.tight_layout()
plt.show()

# Cleaning proposal: retain IDs and ratings, fill missing text with an empty string,
# and preserve original review text because service-specific wording is evidence.
reviews["review_text"] = reviews["review_text"].fillna("").astype(str).str.strip()
print("Cleaning complete: IDs/ratings retained; text normalized only for missing values and whitespace.")

# The designated product-review model is frozen to an immutable revision.
print(f"Model: {MODEL_ID}\nRevision: {MODEL_REVISION}")
classifier = build_classifier()
enriched = enrich_reviews(reviews.drop(columns=["text_length"]), classifier, batch_size=16)
enriched.to_csv(ROOT / "data" / "processed" / "reviews_with_sentiment.csv", index=False)

band_order = ["Positive", "Neutral", "Negative"]
summary = enriched["sentiment_band"].value_counts().reindex(band_order, fill_value=0).to_frame("count")
summary["percent"] = summary["count"] / len(enriched) * 100
print(summary)
print(f"Human average: {enriched['rating'].mean():.2f}/5; model average: {enriched['predicted_stars'].mean():.2f}/5")

# Comparison chart and a reproducible 18-review manual sanity-check sample.
sns.countplot(data=enriched, x="sentiment_band", order=band_order, palette="RdYlGn")
plt.title("Model sentiment bands")
plt.show()
sample = enriched.sample(n=min(18, len(enriched)), random_state=42)[["review_id", "rating", "predicted_stars", "sentiment_band", "review_text"]]
display(sample)

# False negatives: high human rating paired with a 1–2 star model prediction.
false_negatives = enriched[(enriched["rating"] >= 4) & (enriched["predicted_stars"] <= 2)]
print(f"False negatives: {len(false_negatives)} ({len(false_negatives) / len(enriched) * 100:.1f}%)")
display(false_negatives[["review_id", "rating", "predicted_stars", "review_text"]].head(20))
for term in ["wait", "staff", "service", "music", "decor", "price"]:
    print(f"{term}: {enriched['review_text'].str.lower().str.contains(term, regex=False).sum()} reviews")